In [ ]:
# ---------------------------------------------------------------------------------
# Purpose: Create the star schema (GOLD Tier) tables.
# Accept parameters passed from orchestration notebook via dbutils.notebook.run()
# These simulate DAB variables in the bundle deployment 
# ---------------------------------------------------------------------------------

try:
    # Get parameters from dbutils.widgets (passed by dbutils.notebook.run)
    catalog_name = dbutils.widgets.get("catalog_name")
    schema_prefix = dbutils.widgets.get("schema_prefix")
    print(f"Using parameters from orchestration:")
    print(f"  catalog_name: {catalog_name}")
    print(f"  schema_prefix: {schema_prefix}")
except Exception:
    # Fallback to default values if not called from orchestration
    catalog_name = "dev_catalog"
    schema_prefix = "gold_star_hrs"
    print(f"Using default values (not called from orchestration):")
    print(f"  catalog_name: {catalog_name}")
    print(f"  schema_prefix: {schema_prefix}")

In [ ]:
# Create the HRS tables.  Read the SQL file from the Git repository and execute it .
# notes: Create four tables.

# These varibles set for testing only.  Rmove or comment out before deployment.
catalog_name = "dev_catalog"
schema_prefix = "gld_star_hrs"

from pathlib import Path

# Variables are received from the first cell (either from orchestration or defaults)

# Create an array of SQL filesa
tables = [
    "create_dim_hrs_wave.sql",
    "create_dim_hrs_cohort.sql",
    "create_fact_bmi_race_gender.sql"
    "create_fact_hrs_bmi_stats.sql"
]

for table in tables:
    print(f"Creating {table}...")
    
    # Read SQL file
    sql_path = Path(f"../../sql/gold_ddl/{table}")
    sql_text = sql_path.read_text()

    # Replace dev_catalog string with DAB catalog value.
    sql_text = sql_text.replace('dev_catalog', f"{catalog_name}")
    
    # Split and execute statements with parameter binding
    statements = [stmt.strip() for stmt in sql_text.split(';') if stmt.strip()]
    
    for i, stmt in enumerate(statements, 1):
        print(f"  Executing statement {i}/{len(statements)}")
        spark.sql(stmt, args={"catalog_name": catalog_name, "schema_prefix": schema_prefix})
    
    print(f"✓ {table} created")